# Drishti — Abstention Prompt Experiment

**Hypothesis:** the baseline's biggest weakness is a *calibration* problem, not a missing
capability, and prompting alone can fix a large part of it.

Evidence from `01_vizwiz_baseline` (500 samples, stock SmolVLM-Instruct):

| Metric | Value |
|---|---|
| Overall accuracy | 0.308 |
| Ground-truth unanswerable | 244 / 500 (49%) |
| Model said "unanswerable" | 69 / 500 |
| Abstention **precision** | **0.913** |
| Abstention **recall** | **0.258** |

When the model abstains it is right 91% of the time — its judgement is sound. It simply
abstains far too rarely, missing 181 of 244 opportunities. Lifting the unanswerable subset
to 1.0 would move overall accuracy 0.308 → 0.647 (+0.34), against +0.10 for a large gain in
general answering ability.

So: **try the cheap fix before spending days on LoRA.** This notebook is ~30 minutes.
Fine-tuning is Phase 3 and should start from whatever prompt wins here.

### Method

1. Sweep prompt variants on `N_SWEEP=200` samples (fast, ~4 min each).
2. Re-run the winner on the **same 500 samples** as the baseline for a fair comparison.
3. Report precision *and* recall, never accuracy alone — a model that abstains on
   everything scores well on the unanswerable subset while being useless.

Colab: `Runtime → T4 GPU → Run all`.

In [ ]:
%pip install -q -U transformers accelerate datasets einops
import torch, time, string, itertools
from datasets import load_dataset

MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'   # same base as the baseline
N_SWEEP = 200      # variant sweep — keep small, this is a search
N_FULL = 500       # winner confirmation — must match the baseline exactly
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BASELINE = {'overall': 0.308, 'answerable': 0.310, 'unanswerable': 0.306,
            'precision': 0.913, 'recall': 0.258}
print('device:', DEVICE)

In [ ]:
# Same stream, same order, same slice as notebook 01 -- otherwise the comparison is invalid.
stream = load_dataset('lmms-lab/VizWiz-VQA', split='val', streaming=True)
data = list(itertools.islice(stream, N_FULL))

def gt_answers(sample):
    return [a['answer'] if isinstance(a, dict) else a for a in sample['answers']]

# --- scoring: must stay identical to notebook 01 and eval/analyze_results.py ---
ARTICLES = {'a', 'an', 'the'}

def norm(t):
    t = t.lower().strip().translate(str.maketrans('', '', string.punctuation))
    return ' '.join(w for w in t.split() if w not in ARTICLES)

def vizwiz_acc(pred, answers):
    p = norm(pred)
    return min(sum(norm(a) == p for a in answers) / 3.0, 1.0)

def is_unanswerable_gt(answers):
    return sum(norm(a) == 'unanswerable' for a in answers) >= 5

print(f'{len(data)} samples · unanswerable ground truth: '
      f'{sum(is_unanswerable_gt(gt_answers(s)) for s in data)}')

In [ ]:
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as _VisionSeq
except ImportError:
    from transformers import AutoModelForVision2Seq as _VisionSeq

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = _VisionSeq.from_pretrained(MODEL_ID, dtype=torch.float16, device_map=DEVICE)
model.eval()

def answer(img, question, suffix):
    msgs = [{'role': 'user',
             'content': [{'type': 'image'}, {'type': 'text', 'text': question + suffix}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img.convert('RGB')], return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=24, do_sample=False)
    return processor.batch_decode(out, skip_special_tokens=True)[0].split('Assistant:')[-1].strip()

print('model loaded')

## Prompt variants

Each escalates the abstention instruction differently. The risk to watch is **precision
collapse** — a prompt that makes the model abstain indiscriminately will lift recall while
destroying usefulness, and aggregate accuracy alone would not reveal it.

In [ ]:
VARIANTS = {
    # exactly what notebook 01 used -- the control
    'baseline': (
        " Answer in one to three words. If the question cannot be answered"
        " from the image, answer exactly: unanswerable"
    ),

    # tells the model abstention is common, countering a bias toward always answering
    'prior': (
        " Answer in one to three words. About half of these photos are too blurry, too dark,"
        " or badly framed to answer. If this is one of them, answer exactly: unanswerable"
    ),

    # spells out the failure modes instead of leaving 'cannot be answered' abstract
    'criteria': (
        " Answer in one to three words. Answer exactly 'unanswerable' if any of these hold:"
        " the image is blurry or out of focus; it is too dark or too bright; the object asked"
        " about is not visible or is cut off; the text asked about cannot be read clearly."
        " Otherwise answer the question."
    ),

    # pushes the decision threshold rather than describing the criteria
    'cautious': (
        " Answer in one to three words. Only answer if you can see the relevant detail"
        " clearly. If you are guessing at all, answer exactly: unanswerable"
    ),

    # frames the stakes -- the user is blind and cannot sanity-check a wrong answer
    'stakes': (
        " The person asking is blind and cannot check your answer, so a confident wrong"
        " answer is worse than no answer. Answer in one to three words only if the image"
        " clearly shows it. Otherwise answer exactly: unanswerable"
    ),
}

for name, suffix in VARIANTS.items():
    print(f'{name:10s} {len(suffix):4d} chars')

In [ ]:
from tqdm.auto import tqdm

def evaluate(samples, suffix, desc):
    """Run one prompt variant and return per-sample rows plus aggregate metrics."""
    rows = []
    t0 = time.time()
    for s in tqdm(samples, desc=desc, leave=False):
        gt = gt_answers(s)
        try:
            pred = answer(s['image'], s['question'], suffix)
        except Exception as e:
            pred = f'__error__ {e}'
        rows.append({'question': s['question'], 'prediction': pred, 'answers': gt,
                     'acc': vizwiz_acc(pred, gt), 'unanswerable_gt': is_unanswerable_gt(gt)})

    una = [r for r in rows if r['unanswerable_gt']]
    ans = [r for r in rows if not r['unanswerable_gt']]
    said = [r for r in rows if norm(r['prediction']) == 'unanswerable']
    hits = [r for r in said if r['unanswerable_gt']]

    return rows, {
        'overall': sum(r['acc'] for r in rows) / len(rows),
        'answerable': sum(r['acc'] for r in ans) / max(len(ans), 1),
        'unanswerable': sum(r['acc'] for r in una) / max(len(una), 1),
        'precision': len(hits) / len(said) if said else float('nan'),
        'recall': len(hits) / max(len(una), 1),
        'said_una': len(said),
        'secs': time.time() - t0,
    }

sweep_samples = data[:N_SWEEP]
sweep = {}
for name, suffix in VARIANTS.items():
    _, m = evaluate(sweep_samples, suffix, name)
    sweep[name] = m
    print(f"{name:10s} overall {m['overall']:.3f} | abstain P {m['precision']:.3f} "
          f"R {m['recall']:.3f} | said {m['said_una']:3d} | {m['secs']:.0f}s")

In [ ]:
import pandas as pd

df = pd.DataFrame(sweep).T[['overall', 'answerable', 'unanswerable',
                            'precision', 'recall', 'said_una']]
df.index.name = f'variant (N={N_SWEEP})'
display(df.round(3).sort_values('overall', ascending=False))

# Guard against the degenerate win: a prompt can lift overall accuracy purely by abstaining
# on everything. Require precision to stay respectable before accepting a variant.
MIN_PRECISION = 0.70
viable = {k: v for k, v in sweep.items() if v['precision'] >= MIN_PRECISION}

if viable:
    winner = max(viable, key=lambda k: viable[k]['overall'])
else:
    winner = 'baseline'
    print(f'No variant held precision >= {MIN_PRECISION}; falling back to baseline.')

print(f"\nwinner: {winner}")
for k, v in sweep[winner].items():
    if k != 'secs':
        print(f'  {k:14s} {v:.3f}' if isinstance(v, float) else f'  {k:14s} {v}')

In [ ]:
# Confirm the winner on the full 500 -- the same slice the baseline used, so the delta
# is a like-for-like comparison rather than a sample-size artifact.
rows, final = evaluate(data, VARIANTS[winner], f'{winner} (full)')

print(f'PROMPT EXPERIMENT — {winner} vs baseline, N={len(data)}\n')
print(f"{'metric':16s} {'baseline':>9s} {'winner':>9s} {'delta':>9s}")
for k in ('overall', 'answerable', 'unanswerable', 'precision', 'recall'):
    b, w = BASELINE[k], final[k]
    print(f'{k:16s} {b:9.3f} {w:9.3f} {w - b:+9.3f}')

print(f"\nabstained on {final['said_una']}/{len(data)} "
      f"(baseline: 69) · {final['secs']:.0f}s total")

In [ ]:
# Save in the same schema as the baseline CSV so eval/analyze_results.py works unchanged.
out = pd.DataFrame([{'question': r['question'], 'prediction': r['prediction'],
                     'latency_s': 0.0, 'acc': r['acc'],
                     'unanswerable_gt': r['unanswerable_gt'],
                     'gt_sample': '; '.join(r['answers'][:3])} for r in rows])
fname = f'vizwiz_prompt_{winner}_results.csv'
out.to_csv(fname, index=False)
print('saved', fname, '-- download into eval/results/ and run eval/analyze_results.py --csv')

# Cases the winner still misses: ground truth unanswerable, model answered anyway.
still_missed = [r for r in rows if r['unanswerable_gt'] and norm(r['prediction']) != 'unanswerable']
print(f'\nstill guessing on {len(still_missed)} unanswerable questions (baseline: 181):')
for r in still_missed[:10]:
    print(f"  {r['question'][:58]:60s} -> {r['prediction'][:30]}")

## Record for the report

1. Paste the sweep table and the final comparison into `docs/BUILD_PLAN.md` and add a
   decision entry for the chosen prompt.
2. Download the CSV into `eval/results/` — it is versioned deliberately (`DEC-015`).
3. Update the winning suffix in `notebooks/01_vizwiz_baseline.ipynb` so future baseline
   re-runs use it, and note in the report that the comparison was re-based.

### How to read the outcome

**If a variant wins clearly** — prompting recovered part of the +0.34 headroom for zero
training cost. Phase 3 fine-tuning then starts from this prompt and targets what remains,
which is a stronger result than fine-tuning from an untuned prompt and claiming the whole
gain.

**If nothing beats the baseline** — that is also a real finding, and it strengthens the case
for fine-tuning: the behaviour is baked into the weights and instructions cannot shift it.
Report it rather than discarding it.

**If recall rises but precision collapses** — the model was pushed into abstaining
indiscriminately. Aggregate accuracy may still look better while the system becomes useless,
which is exactly why `MIN_PRECISION` gates the winner selection. Say so explicitly in the
report; it demonstrates the metric was chosen with the failure mode in mind.